In [1]:
#set the environment variable for Groq API key
import os

if "GROQ_API_KEY" not in os.environ:
    print("Warning: GROQ_API_KEY environment variable not set")


In [2]:
#Now llm setup
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.5)


In [6]:
#llm schema for the evaluator eg: jokes funny or not

from pydantic import BaseModel,Field
from typing import List,Literal

class llm_schema(BaseModel):
    funny_flag : Literal["funny","not_funny"] = Field(description="Whether the joke is funny or not")
    feedback : str = Field(description="Feeback about the jokes to improve")

llm_with_schema = llm.with_structured_output(llm_schema)
response = llm_with_schema.invoke("Why do we tell actors to break a leg?  Because every play has a cast!")
print(response)


funny_flag='funny' feedback='Good play on words with cast and play!'


In [9]:
#Graph schema 
from typing import TypedDict,List
class Graph_schema(TypedDict):
    topic : str
    jokes: str
    funny_flag: []
    feedback: str
    max_iteration: int

Create a Node


In [ ]:
#generate node takes the topic then generate the jokes if state has feedback  and max_iteration is less than 5 then regenerate the jokes until 5 iteration then end
from langchain_core.prompts import ChatPromptTemplate

def generate_node(state: Graph_schema) -> Graph_schema:

    topic = state["topic"]

    if state["feedbeck"] and (0<state["max_iteration"] < 5) :
        jokes = llm.invoke(f"Modify the jokes {state["jokes"]} on the basis of the feedback is {state["feedback"]}")

    else:
        jokes = llm.invoke(f"create a jokes on the topic of {topic}")

    #update jokes
    state["jokes"] = jokes

    return state


In [11]:
#create a evaluator node
#evaluator node analyze the jokes whether it is funny,not funny and also give feedback generated by generated node 

def evaluator_node(state: Graph_schema) -> Graph_schema:

    jokes = state["jokes"]

    prompt = ChatPromptTemplate.from_messages([
        ("system","You are a gudge of the comedy show,your job is evaluate the joke and provide the feedback."),
        ("user",f"observe the joke: {jokes} and evaluate funny or not then provide the feedback")]
    )

    #define chain
    chain = prompt | llm_with_schema
    response = chain.invoke({"jokes":jokes})

    #update funny ,not funny and feedback
    state["funny_flag"] = response.funny_flag
    state["feedback"] = response.feedback

    return state

